# Characterizing the gap: where the signal lives, and Dr. Miller's same-answer test

EE 456 honors option. Follows `replication.ipynb` (same labels, same hard set: confident, stable under resampling, judge-wrong). Two questions, both answered from activations already saved during the pilot, with no new model runs:

1. **Position check.** Is the hallucination signal already in the model's state when it finishes reading the question (*last prompt token*), or does it appear only once the model has committed to its first answer word (*first answer token*: the state at the position where that word is fed back in)? If the second is much stronger, a "hallucination state" forms while answering. If they are about equal, the signal mostly tracks how hard the question is.
2. **Same-answer test (Miller's strong form).** Take a hallucination that answers R_H, and correct answers elsewhere in the data whose answer is also R_H (the model says "France" wrongly for one question and rightly for others). Is the hallucinated prompt's activation orthogonal to those correct prompts', more than they are to each other?

Shared code: `src/pilot/records.py` (loading), `src/pilot/anomaly.py` (null), `src/pilot/same_answer.py` (test), with tests in `tests/`.

In [1]:
import json, sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src" / "pilot"))
from anomaly import SCORE_NAMES, crossfit_pvalues
from normalize import normalize
from records import load_acts, load_run
from same_answer import centered_cosine_matrix, permutation_test, same_answer_groups

FIG_DIR = ROOT / "results" / "figs_gap"; FIG_DIR.mkdir(parents=True, exist_ok=True)
INK, MUTED, RULE, PAPER, ACCENT = "#26251f", "#6f6f69", "#c9c6bd", "#eeece6", "#bd6593"
SCORE_COLORS = {"mahalanobis": "#4a7c8c", "pca_residual": "#b08a3c", "knn": MUTED, "probe": ACCENT}
SCORE_LABELS = {"mahalanobis": "Mahalanobis", "pca_residual": "PCA residual", "knn": "kNN", "probe": "supervised probe"}
POS_LABELS = {"last_prompt": "last prompt token", "first_answer": "first answer token"}
plt.rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": MUTED,
    "text.color": INK, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": RULE, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "font.family": "sans-serif", "font.size": 10, "axes.titlesize": 12, "axes.titlelocation": "left",
    "legend.frameon": False, "figure.dpi": 110,
})

ALPHA, N_COMPONENTS, KNN_K, N_FOLDS, SEED = 0.05, 20, 10, 5, 0  # same settings as replication.ipynb
MIN_CLASS, N_BOOT, N_PERM = 20, 1000, 10000
RUNS = {"Qwen3-0.6B": "pilot_qwen3_0.6b", "Qwen3-4B": "pilot_qwen3_4b"}
POSITIONS = ("last_prompt", "first_answer")
# Layers fixed before looking at the same-answer test: the probe's best layer and the anomaly
# scores' held-out layer from replication.ipynb (Section 3b).
PRESET_LAYERS = {"Qwen3-4B": (20, 24)}

REC = {m: load_run(ROOT, r) for m, r in RUNS.items()}
OOF = {m: np.load(ROOT / "results" / r / "oof_scores_judge.npz") for m, r in RUNS.items()}
PROBE = {m: {pos: OOF[m][f"probe_{pos}"] for pos in POSITIONS} for m in RUNS}
# The probe cache stores no row ids, so only its length can be checked here; it was built from the same
# load rules in lit_review.ipynb (LABELS=judge). Rerun that notebook after any relabel.
for m, d in REC.items():
    if PROBE[m]["last_prompt"].shape[1] != len(d): raise ValueError(f"{m}: probe scores do not line up with the records")
pd.DataFrame({m: {"answers": len(d), "hard set wrong / total": f"{int(((d.correct == 0) & d.hard).sum())} / {int(d.hard.sum())}",
                  "confident wrong / total": f"{int(((d.correct == 0) & d.confident).sum())} / {int(d.confident.sum())}"}
              for m, d in REC.items()}).T

,answers,hard set wrong / total,confident wrong / total
Qwen3-0.6B,1983,63 / 81,544 / 633
Qwen3-4B,1989,174 / 773,367 / 1048


## 1. Position check

Same procedure as `replication.ipynb` at both positions: cross-fit conformal p-values from a null fit on all correct answers, and the out-of-fold supervised probe (from `lit_review.ipynb`, judge labels). Last-prompt p-values are read from the replication cache; first-answer p-values are computed here and cached in `results/<run>/anomaly_pvals_first_answer_judge.npz`.

In [2]:
FORCE_RECOMPUTE = False

def signature(model):
    d = REC[model]
    return np.array(json.dumps({"qids": list(d.qid), "pool": list(map(bool, d.correct == 1)),
                                "params": [N_COMPONENTS, KNN_K, N_FOLDS, SEED]}))

def first_answer_pvalues(model, run):
    cache = ROOT / "results" / run / "anomaly_pvals_first_answer_judge.npz"; sig = signature(model)
    if cache.exists() and not FORCE_RECOMPUTE:
        z = np.load(cache)
        if str(z["signature"]) == str(sig): return {k: z[k] for k in SCORE_NAMES}
        print(f"{model}: cache is stale, recomputing", flush=True)
    d = REC[model]; acts = load_acts(ROOT, run, d.row.values, "first_answer"); pool = (d.correct == 1).values
    per_layer = [crossfit_pvalues(acts[:, l, :].astype(np.float32), pool, N_COMPONENTS, KNN_K, N_FOLDS, SEED)
                 for l in range(acts.shape[1])]
    out = {name: np.stack([p[name] for p in per_layer]) for name in SCORE_NAMES}
    np.savez_compressed(cache, signature=sig, **out); print(f"{model}: first-answer p-values computed", flush=True)
    return out

def last_prompt_pvalues(model, run):
    z = np.load(ROOT / "results" / run / "anomaly_pvals_judge.npz")
    sig = json.loads(str(z["signature"])); d = REC[model]
    if sig["qids"] != list(d.qid) or sig["pools"]["all correct"] != list(map(bool, d.correct == 1)):
        raise ValueError(f"{model}: replication.ipynb's cache was built on different rows or labels; rerun it")
    return {name: z[f"all correct|{name}"] for name in SCORE_NAMES}

# SCORES[model][position][method] -> (n_layers, n_answers); higher = more typical = predicted correct
SCORES = {m: {"last_prompt": {**last_prompt_pvalues(m, r), "probe": PROBE[m]["last_prompt"]},
              "first_answer": {**first_answer_pvalues(m, r), "probe": PROBE[m]["first_answer"]}} for m, r in RUNS.items()}

In [3]:
def auroc(y, s):
    return float(roc_auc_score(y, s)) if len(np.unique(y)) == 2 else float("nan")

def evaluable(d, sel):
    return min(int((d.correct[sel] == 1).sum()), int((d.correct[sel] == 0).sum())) >= MIN_CLASS

PANELS = [(m, sub) for m, d in REC.items() for sub in ("confident", "hard") if evaluable(d, d[sub])]
METHODS = ("probe", "knn", "pca_residual", "mahalanobis")
fig, axes = plt.subplots(1, len(PANELS), figsize=(5.2 * len(PANELS), 4.2), sharey=True)
for ax, (m, sub) in zip(axes, PANELS):
    d = REC[m]; sel = d[sub].values; y = d.correct.values[sel]
    for name in METHODS:
        for pos, ls in zip(POSITIONS, ("-", "--")):
            curve = [auroc(y, s[sel]) for s in SCORES[m][pos][name]]
            ax.plot(curve, ls=ls, color=SCORE_COLORS[name], lw=2 if name == "probe" else 1.4,
                    label=f"{SCORE_LABELS[name]}, {POS_LABELS[pos]}")
    ax.axhline(0.5, color=MUTED, lw=0.8, ls=":")
    ax.set_title(f"{m}: {'hard set' if sub == 'hard' else 'confident answers'} (n={int(sel.sum())})", fontsize=10.5); ax.set_xlabel("layer")
axes[0].set_ylabel("AUROC (correct vs wrong)"); axes[-1].legend(fontsize=7.5, loc="lower left", ncol=2)
fig.tight_layout(); fig.savefig(FIG_DIR / "01_position_auroc_by_layer.png", dpi=160, bbox_inches="tight"); plt.show()

/var/folders/1f/rj7xg3297314237bcmb784h00000gn/T/ipykernel_99649/3734177495.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(FIG_DIR / "01_position_auroc_by_layer.png", dpi=160, bbox_inches="tight"); plt.show()


**Read:** solid lines (last prompt token) sit at or above dashed lines (first answer token) almost everywhere. The supervised probe reaches about the same peak at both positions, so committing to the first answer word adds no hallucination signal the model did not already carry at the end of the question.

The anomaly scores at the first answer token behave very differently: in the early 4B layers they fall to about 0.37, well *below* 0.5. There, hallucinations look *more* typical than correct answers. At layer 0 that activation is just the embedding of the first answer token, so this is about which word the answer starts with: hallucinations tend to start with common words that many other answers also start with, while correct answers start with rarer ones. It is a first hint for the "popular substitute" test in Phase 3, not evidence by itself.

### Held-out comparison of the two positions

As in `replication.ipynb` Section 3b: each method's layer is chosen on the first 500 questions' hard set and scored once on questions 501-2000. The last column is the held-out AUROC difference (first answer minus last prompt), with a paired bootstrap CI over the same held-out answers. 4B only, since the 0.6B hard set is too small to split.

In [4]:
def choose_layer(S, sel, y):
    auc = np.array([auroc(y, s[sel]) for s in S]); return int(np.nanargmax(auc)), float(np.nanmax(auc))

def paired_boot_diff(y, a, b, seed=SEED):
    rng = np.random.default_rng(seed); diffs = []
    for _ in range(N_BOOT):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) == 2: diffs.append(roc_auc_score(y[i], a[i]) - roc_auc_score(y[i], b[i]))
    return np.percentile(diffs, [2.5, 97.5])

rows = []
for m, d in REC.items():
    if not (evaluable(d, d.hard_explore) and evaluable(d, d.hard_holdout)): continue
    ex, ho = d.hard_explore.values, d.hard_holdout.values; y_ex, y_ho = d.correct.values[ex], d.correct.values[ho]
    for name in METHODS:
        held = {}
        for pos in POSITIONS:
            S = SCORES[m][pos][name]; l, a_ex = choose_layer(S, ex, y_ex)
            held[pos] = S[l][ho]
            rows.append({"model": m, "method": SCORE_LABELS[name], "position": POS_LABELS[pos], "layer chosen": l,
                         "AUROC first 500": a_ex, "AUROC held-out": auroc(y_ho, held[pos])})
        lo, hi = paired_boot_diff(y_ho, held["first_answer"], held["last_prompt"])
        rows[-1]["held-out diff (answer - prompt)"] = f"{auroc(y_ho, held['first_answer']) - auroc(y_ho, held['last_prompt']):+.3f} ({lo:+.2f} to {hi:+.2f})"
POSITION_CONFIRM = pd.DataFrame(rows).fillna("")
POSITION_CONFIRM.style.format({"AUROC first 500": "{:.3f}", "AUROC held-out": "{:.3f}"}).hide(axis="index")

model,method,position,layer chosen,AUROC first 500,AUROC held-out,held-out diff (answer - prompt)
Qwen3-4B,supervised probe,last prompt token,24,0.764,0.811,
Qwen3-4B,supervised probe,first answer token,25,0.783,0.764,-0.046 (-0.09 to -0.00)
Qwen3-4B,kNN,last prompt token,20,0.730,0.653,
Qwen3-4B,kNN,first answer token,22,0.590,0.511,-0.141 (-0.20 to -0.08)
Qwen3-4B,PCA residual,last prompt token,20,0.733,0.631,
Qwen3-4B,PCA residual,first answer token,20,0.604,0.512,-0.119 (-0.18 to -0.06)
Qwen3-4B,Mahalanobis,last prompt token,20,0.656,0.528,
Qwen3-4B,Mahalanobis,first answer token,0,0.632,0.574,+0.046 (-0.04 to +0.14)


**Read:** on the 1,500 held-out questions the last prompt token wins for every method that works. The probe scores 0.81 there against 0.76 at the first answer token (difference -0.05, CI -0.09 to 0.00). kNN and PCA residual lose all signal at the first answer token (0.51), 0.12-0.14 below the last prompt token, with CIs clear of zero. Mahalanobis picks layer 0 at the first answer token, which is the first-word identity effect above, not a hallucination state.

So the signal is about the question and what the model knows when it starts answering, not a state that appears once it commits to an answer.

## 2. Dr. Miller's same-answer test (4B)

**Groups.** For each 4B hard-set hallucination, the correct answers (any question, judge labels) whose normalized answer text is the same. 44 of 174 hallucinations have at least one; the test uses the groups with at least two, because with one correct prompt, swapping which prompt is "the hallucination" leaves the cosine unchanged.

**Cosines.** Activations are centered on the mean of all correct answers at that layer and position. Raw activations share a large common offset, which makes every raw cosine 0.3-0.7. After centering, two unrelated prompts sit near 0.

**Test.** Statistic: the hallucination's mean cosine to its group's correct prompts, averaged over groups. Null: within each group, the hallucination label is equally likely to belong to any member, so it is reassigned at random 10,000 times. One-sided p: how often a random assignment is at least as unaligned as the real one. The test is run at the **last prompt token** only, at the two layers fixed in advance (20 and 24), with a Holm correction over those two. The first answer token is shown in the figure for context but not tested: every member of a group is fed the same answer word at that position, so the cosine there mostly measures that shared word and cannot separate a hallucination from a correct answer.

**Caveats.** (1) Several hallucinations share one answer ("France" three times), so their groups share correct prompts. The permutation treats them as independent, which makes p somewhat too small; since nothing is significant, this only strengthens the null reading. (2) The black "correct ↔ correct" line averages over the other correct prompts only, while the test's null also includes the hallucination, so the gap in the figure looks a little larger than the gap the test measures.

In [5]:
m, run = "Qwen3-4B", RUNS["Qwen3-4B"]; d = REC[m]
answers = d.answer.map(normalize).values
is_h, is_c = (d.hard & (d.correct == 0)).values, (d.correct == 1).values
ALL_GROUPS = same_answer_groups(answers, is_h, is_c, min_correct=1)
GROUPS = [g for g in ALL_GROUPS if len(g.correct) >= 2]
group_table = pd.DataFrame([{"question": d.question[g.hallucination], "model answer": d.answer[g.hallucination],
                             "gold": d.gold[g.hallucination], "correct prompts with this answer": len(g.correct)} for g in ALL_GROUPS])
print(f"{len(ALL_GROUPS)} hallucinations with a same-answer correct prompt; {len(GROUPS)} with at least two (used in the test)")
group_table.sort_values("correct prompts with this answer", ascending=False).head(15)

44 hallucinations with a same-answer correct prompt; 24 with at least two (used in the test)


,question,model answer,gold,correct prompts with this answer
22,The UEFA (Union of European Football Associati...,France,Switzerland,5
3,Marc Dutroux hit the headlines over a 'house o...,France,Belgium,5
32,The World Bog Snorkelling Championships are he...,France,Wales,5
27,"In cricket, which nation won the 2009 Ashes se...",Australia,England,4
12,What was founded by Ferdinand II of Aragon and...,Spain,Spanish Inquisition,4
13,Captain Corelli's mandolin is a book a film se...,Italy,Greece,4
29,Michaelmas Day is celebrated on the 29th of wh...,November,September,3
20,The Wang River is in which Asian country?,China,Thailand,3
2,Which artist David was born in Bradford UK?,David Bowie,Hockney,2
18,What colour is the cross on the national flag ...,Red,White,2


In [6]:
rows_needed = sorted({i for g in GROUPS for i in (g.hallucination, *g.correct)})
local = {r: k for k, r in enumerate(rows_needed)}
LOCAL_GROUPS = [type(g)(hallucination=local[g.hallucination], correct=tuple(local[c] for c in g.correct)) for g in GROUPS]
# reference: cosine between correct prompts with *different* answers, drawn from the same rows
diff_pairs = [(local[a], local[b]) for i, ga in enumerate(GROUPS) for gb in GROUPS[i + 1:]
              if answers[ga.correct[0]] != answers[gb.correct[0]] for a, b in [(ga.correct[0], gb.correct[0])]]

SAME = []
for pos in POSITIONS:
    acts = load_acts(ROOT, run, d.row.values, pos)
    for l in range(1, acts.shape[1]):  # layer 0 is the token embedding, identical for every prompt at these positions
        center = acts[is_c, l, :].astype(np.float64).mean(0)
        cos = centered_cosine_matrix(acts[rows_needed, l, :].astype(np.float64), center)
        r = permutation_test(cos, LOCAL_GROUPS, n_perm=N_PERM, seed=SEED)
        SAME.append({"position": pos, "layer": l, "hallucination cos": r.hallucination_cos, "correct cos": r.correct_cos,
                     "different-answer cos": float(np.mean([cos[a, b] for a, b in diff_pairs])), "p": r.p_value, "groups": r.n_groups})
SAME = pd.DataFrame(SAME)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, pos in zip(axes, POSITIONS):
    g = SAME[SAME.position == pos]
    ax.plot(g.layer, g["correct cos"], color=INK, lw=1.8, label="correct ↔ correct, same answer")
    ax.plot(g.layer, g["hallucination cos"], color=ACCENT, lw=2, label="hallucination ↔ correct, same answer")
    ax.plot(g.layer, g["different-answer cos"], color=MUTED, lw=1.2, ls="--", label="correct ↔ correct, different answers")
    for l in PRESET_LAYERS[m]: ax.axvline(l, color=RULE, lw=1)
    ax.axhline(0, color=MUTED, lw=0.8, ls=":"); ax.set_title(f"{m}, {POS_LABELS[pos]}", fontsize=10.5); ax.set_xlabel("layer")
axes[0].set_ylabel("mean centered cosine"); axes[0].legend(fontsize=8.5)
fig.tight_layout(); fig.savefig(FIG_DIR / "02_same_answer_cosine.png", dpi=160, bbox_inches="tight"); plt.show()

def holm(p):
    order = np.argsort(p); k = len(p)
    adjusted = np.minimum(np.maximum.accumulate((k - np.arange(k)) * p[order]), 1.0)
    out = np.empty(k); out[order] = adjusted; return out

PRESET = SAME[SAME.layer.isin(PRESET_LAYERS[m]) & (SAME.position == "last_prompt")].copy()
PRESET["Holm p"] = holm(PRESET.p.values)
PRESET.assign(position=PRESET.position.map(POS_LABELS)).style.format(
    {c: "{:.3f}" for c in ["hallucination cos", "correct cos", "different-answer cos"]} | {"p": "{:.4f}", "Holm p": "{:.4f}"}).hide(axis="index")

/var/folders/1f/rj7xg3297314237bcmb784h00000gn/T/ipykernel_99649/437937353.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(FIG_DIR / "02_same_answer_cosine.png", dpi=160, bbox_inches="tight"); plt.show()


position,layer,hallucination cos,correct cos,different-answer cos,p,groups,Holm p
last prompt token,20,0.314,0.335,0.051,0.3150,24,0.3150
last prompt token,24,0.348,0.379,0.047,0.0946,24,0.1892


**Read:** Miller's strong form is not supported. After centering, correct prompts with the *same* answer are clearly aligned (0.33-0.38 at the last prompt token, layers 20 and 24), against about 0.05 for correct prompts with different answers: the state at the end of the question already encodes the answer the model is about to give.

The hallucinated prompts are aligned with their same-answer correct prompts almost as strongly (0.31-0.35), far from orthogonal. They sit slightly below the correct-correct line at most last-prompt layers, but neither preset test is significant (layer 20: p = 0.32; layer 24: p = 0.09, Holm 0.19). The first-answer panel is not evidence either way, for the reason given above.

In plain terms: when the model wrongly says "France" for the Belgian question, its internal state looks like the state it has when it rightly says "France". That fits the CHOKE picture of confident, consistent errors. It also means whatever the probe picks up (0.81) is a small direction that these cosines, dominated by the answer itself, barely register.

Limits: 24 groups, several sharing correct prompts, so only a fairly large gap could be detected. Most of the same-answer groups are countries, colours and numbers.